In [1]:
import pandas as pd
import numpy as np
import re
import os
import unicodedata

In [2]:
RAW_DIR = "data/raw"
CLEAN_DIR = "data/cleaned"

os.makedirs(CLEAN_DIR, exist_ok=True)

PEOPLE_FILE = os.path.join(RAW_DIR, "01_people.csv")
EDUCATION_FILE = os.path.join(RAW_DIR, "03_education.csv")
EXPERIENCE_FILE = os.path.join(RAW_DIR, "04_experience.csv")
SKILLS_FILE = os.path.join(RAW_DIR, "05_person_skills.csv")

# Change this filename to whatever your downloaded Coursera CSV is called
COURSERA_FILE = os.path.join(RAW_DIR, "coursera_data.csv")

In [3]:
people = pd.read_csv(PEOPLE_FILE)
education = pd.read_csv(EDUCATION_FILE)
experience = pd.read_csv(EXPERIENCE_FILE)
person_skills = pd.read_csv(SKILLS_FILE)

courses = pd.read_csv(COURSERA_FILE)

In [4]:
datasets = {
    "people": people,
    "education": education,
    "experience": experience,
    "person_skills": person_skills,
    "courses": courses
}

for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(name.upper())
    print(f"Shape: {df.shape}")
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nMissing values:")
    print(df.isna().sum())


PEOPLE
Shape: (54933, 5)

Columns:
['person_id', 'name', 'email', 'phone', 'linkedin']

Missing values:
person_id        0
name           114
email        53340
phone        53100
linkedin     46395
dtype: int64

EDUCATION
Shape: (75999, 5)

Columns:
['person_id', 'institution', 'program', 'start_date', 'location']

Missing values:
person_id          0
institution     1569
program         7761
start_date     21129
location       23256
dtype: int64

EXPERIENCE
Shape: (265404, 6)

Columns:
['person_id', 'title', 'firm', 'start_date', 'end_date', 'location']

Missing values:
person_id         0
title           111
firm           4203
start_date     2262
end_date       2724
location      53055
dtype: int64

PERSON_SKILLS
Shape: (2483376, 2)

Columns:
['person_id', 'skill']

Missing values:
person_id    0
skill        9
dtype: int64

COURSES
Shape: (623, 12)

Columns:
['Unnamed: 0', 'Title', 'Organization', 'Skills', 'Ratings', 'course_url', 'course_students_enrolled', 'course_description'

In [5]:
def clean_column_names(df):
    df = df.copy()
    
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    
    return df


people = clean_column_names(people)
education = clean_column_names(education)
experience = clean_column_names(experience)
person_skills = clean_column_names(person_skills)
courses = clean_column_names(courses)

In [6]:
for name, df in {
    "people": people,
    "education": education,
    "experience": experience,
    "person_skills": person_skills,
    "courses": courses
}.items():
    print(name, "->", df.columns.tolist())

people -> ['person_id', 'name', 'email', 'phone', 'linkedin']
education -> ['person_id', 'institution', 'program', 'start_date', 'location']
experience -> ['person_id', 'title', 'firm', 'start_date', 'end_date', 'location']
person_skills -> ['person_id', 'skill']
courses -> ['unnamed_0', 'title', 'organization', 'skills', 'ratings', 'course_url', 'course_students_enrolled', 'course_description', 'review_count', 'difficulty', 'type', 'duration']


In [7]:
def clean_text(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value)
    
    # Normalize unicode
    value = unicodedata.normalize("NFKC", value)
    
    # Replace newlines/tabs with spaces
    value = re.sub(r"[\r\n\t]+", " ", value)
    
    # Collapse repeated whitespace
    value = re.sub(r"\s+", " ", value)
    
    return value.strip()

In [8]:
people = people[["person_id", "name"]].copy()

people["person_id"] = pd.to_numeric(
    people["person_id"],
    errors="coerce"
).astype("Int64")

people["name"] = people["name"].apply(clean_text)

# Remove rows without an ID
people = people.dropna(subset=["person_id"])

# person_id should uniquely identify a person
people = people.drop_duplicates(subset=["person_id"])

print(people.shape)
print(people.head())

(54933, 2)
   person_id                                               name
0          1                             Database Administrator
1          2                             Database Administrator
2          3                      Oracle Database Administrator
3          4  Amazon Redshift Administrator and ETL Develope...
4          5             Scrum Master Scrum Master Scrum Master


In [9]:
education = education[
    ["person_id", "institution", "program", "location"]
].copy()

education["person_id"] = pd.to_numeric(
    education["person_id"],
    errors="coerce"
).astype("Int64")

for col in ["institution", "program", "location"]:
    education[col] = education[col].apply(clean_text)

education = education.dropna(subset=["person_id"])

# Remove completely duplicated education records
education = education.drop_duplicates()

print(education.shape)
print(education.head())

(75918, 4)
   person_id                       institution  \
0          1              Lead City University   
1          2            lagos state university   
2          3   JNTU - Kakinada, Andhra Pradesh   
3          4         University of Informatics   
4          5  Virginia Commomwealth University   

                                             program                  location  
0                                Bachelor of Science                       NaN  
1                            bsc in computer science                 Lagos, GU  
2  Master of Computer Applications in Science and...  Kakinada, Andhra Pradesh  
3                       Bachelor in Computer Science                 June 2007  
4                                                NaN              Richmond, VA  


In [10]:
experience = experience[
    ["person_id", "title", "firm", "location"]
].copy()

experience["person_id"] = pd.to_numeric(
    experience["person_id"],
    errors="coerce"
).astype("Int64")

for col in ["title", "firm", "location"]:
    experience[col] = experience[col].apply(clean_text)

experience = experience.dropna(subset=["person_id"])

experience = experience.drop_duplicates()

print(experience.shape)
print(experience.head())

(263718, 4)
   person_id                          title                       firm  \
0          1         Database Administrator    Family Private Care LLC   
1          1         Database Administrator                     Incomm   
2          2         Database Administrator  Intercontinental Registry   
3          3  Oracle Database Administrator                  Cognizant   
4          3  Oracle Database Administrator                  Convergys   

               location  
0           Roswell, GA  
1        Alpharetta, GA  
2             Lagos, GU  
3  Hyderabad, Telangana  
4  Hyderabad, Telangana  


In [11]:
person_skills = person_skills[
    ["person_id", "skill"]
].copy()

person_skills["person_id"] = pd.to_numeric(
    person_skills["person_id"],
    errors="coerce"
).astype("Int64")

person_skills["skill"] = person_skills["skill"].apply(clean_text)

person_skills = person_skills.dropna(
    subset=["person_id", "skill"]
)

# Normalize skill matching
person_skills["skill_normalized"] = (
    person_skills["skill"]
    .str.lower()
    .str.strip()
)

# Remove exact duplicate person-skill relationships
person_skills = person_skills.drop_duplicates(
    subset=["person_id", "skill_normalized"]
)

print(person_skills.shape)
print(person_skills.head())

(1873884, 3)
   person_id                    skill         skill_normalized
0          1  Database administration  database administration
1          1                 Database                 database
2          1            Ms sql server            ms sql server
3          1       Ms sql server 2005       ms sql server 2005
4          1               Sql server               sql server


In [12]:
valid_person_ids = set(people["person_id"].dropna())

education = education[
    education["person_id"].isin(valid_person_ids)
].copy()

experience = experience[
    experience["person_id"].isin(valid_person_ids)
].copy()

person_skills = person_skills[
    person_skills["person_id"].isin(valid_person_ids)
].copy()

In [13]:
print("Education orphan records:",
      (~education["person_id"].isin(valid_person_ids)).sum())

print("Experience orphan records:",
      (~experience["person_id"].isin(valid_person_ids)).sum())

print("Skill orphan records:",
      (~person_skills["person_id"].isin(valid_person_ids)).sum())

Education orphan records: 0
Experience orphan records: 0
Skill orphan records: 0


In [14]:
print(courses.columns.tolist())
print(courses.head())

['unnamed_0', 'title', 'organization', 'skills', 'ratings', 'course_url', 'course_students_enrolled', 'course_description', 'review_count', 'difficulty', 'type', 'duration']
   unnamed_0                                  title organization  \
0          0                   Google Cybersecurity       Google   
1          1                  Google Data Analytics       Google   
2          3             Google Project Management:       Google   
3          4                       IBM Data Science          IBM   
4          5  Google Digital Marketing & E-commerce       Google   

                                              skills  ratings  \
0   Network Security, Python Programming, Linux, ...      4.8   
1   Data Analysis, R Programming, SQL, Business C...      4.8   
2   Project Management, Strategy and Operations, ...      4.8   
3   Python Programming, Data Science, Machine Lea...      4.6   
4   Digital Marketing, Marketing, Marketing Manag...      4.8   

                          

In [15]:
for col in courses.select_dtypes(include="object").columns:
    courses[col] = courses[col].apply(clean_text)

/var/folders/wd/jtsz545931z6vjnhzc7f2zwc0000gn/T/ipykernel_4044/1295235715.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in courses.select_dtypes(include="object").columns:


In [16]:
print(courses.columns.tolist())
COURSE_SKILL_COL = "skills"

['unnamed_0', 'title', 'organization', 'skills', 'ratings', 'course_url', 'course_students_enrolled', 'course_description', 'review_count', 'difficulty', 'type', 'duration']


In [17]:
print(courses[COURSE_SKILL_COL].head(10))

0    Network Security, Python Programming, Linux, C...
1    Data Analysis, R Programming, SQL, Business Co...
2    Project Management, Strategy and Operations, L...
3    Python Programming, Data Science, Machine Lear...
4    Digital Marketing, Marketing, Marketing Manage...
5    Python Programming, Microsoft Excel, Data Visu...
6    Computer Networking, Network Architecture, Net...
7    Machine Learning, Machine Learning Algorithms,...
8    User Experience, User Experience Design, User ...
9    DevOps, Software Engineering, Cloud Computing,...
Name: skills, dtype: str


In [19]:
courses["skills_normalized"] = (
    courses[COURSE_SKILL_COL]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

In [20]:
print(courses.columns.tolist())

['unnamed_0', 'title', 'organization', 'skills', 'ratings', 'course_url', 'course_students_enrolled', 'course_description', 'review_count', 'difficulty', 'type', 'duration', 'skills_normalized']


In [21]:
for col in [
    "ratings",
    "course_students_enrolled",
    "review_count"
]:
    if col in courses.columns:
        courses[col] = (
            courses[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("+", "", regex=False)
            .replace(["nan", "None", ""], np.nan)
        )
        
        courses[col] = pd.to_numeric(
            courses[col],
            errors="coerce"
        )

In [22]:
if "course_url" in courses.columns:
    courses = courses.drop_duplicates(
        subset=["course_url"],
        keep="first"
    )
elif "url" in courses.columns:
    courses = courses.drop_duplicates(
        subset=["url"],
        keep="first"
    )
elif "course_title" in courses.columns:
    courses = courses.drop_duplicates(
        subset=["course_title"],
        keep="first"
    )
elif "name" in courses.columns:
    courses = courses.drop_duplicates(
        subset=["name"],
        keep="first"
    )

In [23]:
people = people.dropna(
    subset=["person_id"]
)

In [24]:
education = education.dropna(
    subset=["person_id"]
)

experience = experience.dropna(
    subset=["person_id"]
)

person_skills = person_skills.dropna(
    subset=["person_id", "skill"]
)

In [25]:
if "title" in courses.columns:
    courses = courses.dropna(subset=["title"])

In [26]:
def dataset_report(name, df):
    print(f"\n{'='*70}")
    print(f"{name.upper()} DATA QUALITY REPORT")
    print(f"{'='*70}")
    
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    
    print("\nDuplicate rows:", df.duplicated().sum())
    
    print("\nMissing values:")
    missing = df.isna().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    
    report = pd.DataFrame({
        "missing_count": missing,
        "missing_percentage": missing_pct
    })
    
    print(report[report["missing_count"] > 0])

In [27]:
dataset_report("People", people)
dataset_report("Education", education)
dataset_report("Experience", experience)
dataset_report("Person Skills", person_skills)
dataset_report("Courses", courses)


PEOPLE DATA QUALITY REPORT
Rows: 54933
Columns: 2

Duplicate rows: 0

Missing values:
      missing_count  missing_percentage
name            114                0.21

EDUCATION DATA QUALITY REPORT
Rows: 75918
Columns: 4

Duplicate rows: 0

Missing values:
             missing_count  missing_percentage
institution           1569                2.07
program               7755               10.21
location             23235               30.61

EXPERIENCE DATA QUALITY REPORT
Rows: 263718
Columns: 4

Duplicate rows: 0

Missing values:
          missing_count  missing_percentage
title               105                0.04
firm               4161                1.58
location          52464               19.89

PERSON SKILLS DATA QUALITY REPORT
Rows: 1873884
Columns: 3

Duplicate rows: 0

Missing values:
Empty DataFrame
Columns: [missing_count, missing_percentage]
Index: []

COURSES DATA QUALITY REPORT
Rows: 404
Columns: 13

Duplicate rows: 0

Missing values:
                          missing

In [28]:
print("\nREFERENTIAL INTEGRITY")
print("-" * 50)

print(
    "Education → People:",
    education["person_id"].isin(people["person_id"]).all()
)

print(
    "Experience → People:",
    experience["person_id"].isin(people["person_id"]).all()
)

print(
    "Skills → People:",
    person_skills["person_id"].isin(people["person_id"]).all()
)


REFERENTIAL INTEGRITY
--------------------------------------------------
Education → People: True
Experience → People: True
Skills → People: True


In [29]:
people.to_csv(
    os.path.join(CLEAN_DIR, "people_cleaned.csv"),
    index=False
)

education.to_csv(
    os.path.join(CLEAN_DIR, "education_cleaned.csv"),
    index=False
)

experience.to_csv(
    os.path.join(CLEAN_DIR, "experience_cleaned.csv"),
    index=False
)

person_skills.to_csv(
    os.path.join(CLEAN_DIR, "person_skills_cleaned.csv"),
    index=False
)

courses.to_csv(
    os.path.join(CLEAN_DIR, "courses_cleaned.csv"),
    index=False
)

In [30]:
for name, df in {
    "people": people,
    "education": education,
    "experience": experience,
    "person_skills": person_skills,
    "courses": courses
}.items():
    print("\n" + "="*70)
    print(name.upper())
    print("="*70)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nDtypes:")
    print(df.dtypes)
    print("\nSample:")
    display(df.head(3))


PEOPLE
Shape: (54933, 2)
Columns: ['person_id', 'name']

Dtypes:
person_id    Int64
name           str
dtype: object

Sample:


,person_id,name
0,1,Database Administrator
1,2,Database Administrator
2,3,Oracle Database Administrator



EDUCATION
Shape: (75918, 4)
Columns: ['person_id', 'institution', 'program', 'location']

Dtypes:
person_id      Int64
institution      str
program          str
location         str
dtype: object

Sample:


,person_id,institution,program,location
0,1,Lead City University,Bachelor of Science,NaN
1,2,lagos state university,bsc in computer science,"Lagos, GU"
2,3,"JNTU - Kakinada, Andhra Pradesh",Master of Computer Applications in Science and...,"Kakinada, Andhra Pradesh"



EXPERIENCE
Shape: (263718, 4)
Columns: ['person_id', 'title', 'firm', 'location']

Dtypes:
person_id    Int64
title          str
firm           str
location       str
dtype: object

Sample:


,person_id,title,firm,location
0,1,Database Administrator,Family Private Care LLC,"Roswell, GA"
1,1,Database Administrator,Incomm,"Alpharetta, GA"
2,2,Database Administrator,Intercontinental Registry,"Lagos, GU"



PERSON_SKILLS
Shape: (1873884, 3)
Columns: ['person_id', 'skill', 'skill_normalized']

Dtypes:
person_id           Int64
skill                 str
skill_normalized      str
dtype: object

Sample:


,person_id,skill,skill_normalized
0,1,Database administration,database administration
1,1,Database,database
2,1,Ms sql server,ms sql server



COURSES
Shape: (404, 13)
Columns: ['unnamed_0', 'title', 'organization', 'skills', 'ratings', 'course_url', 'course_students_enrolled', 'course_description', 'review_count', 'difficulty', 'type', 'duration', 'skills_normalized']

Dtypes:
unnamed_0                     int64
title                           str
organization                    str
skills                          str
ratings                     float64
course_url                      str
course_students_enrolled    float64
course_description              str
review_count                float64
difficulty                      str
type                            str
duration                        str
skills_normalized               str
dtype: object

Sample:


,unnamed_0,title,organization,skills,ratings,course_url,course_students_enrolled,course_description,review_count,difficulty,type,duration,skills_normalized
0,0,Google Cybersecurity,Google,"Network Security, Python Programming, Linux, C...",4.8,https://www.coursera.org/professional-certific...,700909.0,Google Cloud Fundamentals: Core Infrastructure...,NaN,Beginner,Professional Certificate,3 - 6 Months,"network security, python programming, linux, c..."
1,1,Google Data Analytics,Google,"Data Analysis, R Programming, SQL, Business Co...",4.8,https://www.coursera.org/professional-certific...,229865.0,Prepare for a new career in the high-growth fi...,NaN,Beginner,Professional Certificate,3 - 6 Months,"data analysis, r programming, sql, business co..."
2,3,Google Project Management:,Google,"Project Management, Strategy and Operations, L...",4.8,https://www.coursera.org/professional-certific...,29702.0,Prepare-se para uma nova carreira no campo de ...,NaN,Beginner,Professional Certificate,3 - 6 Months,"project management, strategy and operations, l..."


In [31]:
print("Skills per person:")
print(
    person_skills
    .groupby("person_id")
    .size()
    .describe()
)

print("\nExperience records per person:")
print(
    experience
    .groupby("person_id")
    .size()
    .describe()
)

print("\nEducation records per person:")
print(
    education
    .groupby("person_id")
    .size()
    .describe()
)

Skills per person:
count    54858.000000
mean        34.158810
std         30.624666
min          1.000000
25%         15.000000
50%         25.000000
75%         42.000000
max        320.000000
dtype: float64

Experience records per person:
count    54933.000000
mean         4.800721
std          2.704740
min          1.000000
25%          3.000000
50%          4.000000
75%          6.000000
max         33.000000
dtype: float64

Education records per person:
count    48075.000000
mean         1.579158
std          0.797658
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         15.000000
dtype: float64
